# SHAP for Explainability in Machine Learning
### *Chapter 3 — XAI Techniques | Explainable AI in Medical Systems*

---

**SHAP** (SHapley Additive exPlanations) is the most widely used XAI framework in the medical literature, appearing in 67.9% of the 644 clinical papers reviewed in Chapter 4. It is grounded in cooperative game theory: each feature receives a Shapley value representing its **fair contribution** to a specific prediction relative to the average model output.

This notebook demonstrates SHAP across three different model types, mirrors the workflow used in the majority of clinical XAI papers, and covers all major visualisation types. The dataset used — the **Wisconsin Breast Cancer Dataset** — is a standard benchmark for binary classification in oncology, making the examples directly relevant to the medical domain.

---

## Contents

1. [Setup and dataset](#1)
2. [Model training and evaluation](#2)
3. [SHAP with TreeExplainer — Gradient Boosting Machine](#3)
   - 3.1 Summary plot (dot)
   - 3.2 Summary plot (bar) — global feature importance
   - 3.3 Beeswarm plot
   - 3.4 Heatmap
   - 3.5 Waterfall plot — local explanation
   - 3.6 Force plot — local explanation
   - 3.7 Dependence plot
   - 3.8 Decision plot — multiple instances
4. [SHAP with TreeExplainer — Random Forest](#4)
5. [SHAP with LinearExplainer — Logistic Regression](#5)
6. [Summary: SHAP explainer types and when to use them](#6)

<a id='1'></a>
## 1. Setup and dataset

Install dependencies if needed and load the Wisconsin Breast Cancer Dataset. The task is binary classification: predict whether a tumour is **malignant (0)** or **benign (1)** from 30 numeric features derived from digitised images of fine needle aspirates.

In [ ]:
# Install required packages (run once)
# !pip install shap scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

warnings.filterwarnings('ignore')
shap.initjs()  # enables interactive JS plots in Jupyter

print(f"SHAP version : {shap.__version__}")

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
data   = load_breast_cancer()
X      = pd.DataFrame(data.data, columns=data.feature_names)
y      = pd.Series(data.target, name='diagnosis')  # 0 = malignant, 1 = benign

print("Dataset shape :", X.shape)
print("Features      :", X.shape[1])
print()
print("Class distribution:")
print(y.value_counts().rename({0: 'malignant (0)', 1: 'benign (1)'}).to_string())

X.head(3)

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set : {X_train.shape[0]} samples")
print(f"Test set     : {X_test.shape[0]} samples")

<a id='2'></a>
## 2. Model training and evaluation

We train three model families used in the medical XAI literature:

| Model | SHAP explainer | Explanation type |
|---|---|---|
| Gradient Boosting Machine (GBM) | `TreeExplainer` | Exact, model-specific |
| Random Forest (RF) | `TreeExplainer` | Exact, model-specific |
| Logistic Regression (LR) | `LinearExplainer` | Exact, model-specific |

> **Note:** For truly black-box models (e.g., SVMs, neural networks) a `KernelExplainer` is used instead — it is model-agnostic but computationally slower.

In [ ]:
# ── Gradient Boosting Machine ─────────────────────────────────────────────────
gbm = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
gbm.fit(X_train, y_train)

auc_gbm = roc_auc_score(y_test, gbm.predict_proba(X_test)[:, 1])
print(f"GBM  — AUC-ROC: {auc_gbm:.4f}")
print(classification_report(y_test, gbm.predict(X_test),
                             target_names=['malignant', 'benign']))

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print(f"RF   — AUC-ROC: {auc_rf:.4f}")
print(classification_report(y_test, rf.predict(X_test),
                             target_names=['malignant', 'benign']))

In [ ]:
# ── Logistic Regression (requires standardisation) ────────────────────────────
scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Keep column names for SHAP
X_train_sc_df = pd.DataFrame(X_train_sc, columns=X_train.columns)
X_test_sc_df  = pd.DataFrame(X_test_sc,  columns=X_test.columns)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

auc_lr = roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:, 1])
print(f"LR   — AUC-ROC: {auc_lr:.4f}")
print(classification_report(y_test, lr.predict(X_test_sc),
                             target_names=['malignant', 'benign']))

<a id='3'></a>
## 3. SHAP with TreeExplainer — Gradient Boosting Machine

`TreeExplainer` uses an exact, polynomial-time algorithm that exploits the tree structure of gradient boosted and random forest models. It is the fastest and most accurate SHAP explainer and is the one used in the vast majority of clinical XAI papers.

### How to interpret SHAP values

- A **positive** SHAP value pushes the prediction **toward the positive class** (benign)
- A **negative** SHAP value pushes the prediction **toward the negative class** (malignant)
- The **base value** (expected value) is the average model output across all training samples
- `individual prediction = base value + sum of all SHAP values`

In [ ]:
# ── Compute SHAP values ───────────────────────────────────────────────────────
explainer_gbm = shap.TreeExplainer(gbm)

# shap_values: numpy array (n_samples x n_features) — used with legacy plot API
shap_values_gbm = explainer_gbm.shap_values(X_test)

# Explanation object — used with new plot API (waterfall, beeswarm, heatmap)
shap_exp_gbm = explainer_gbm(X_test)

# Base value (expected model output on training data)
base_value_gbm = explainer_gbm.expected_value[0]

print(f"SHAP values shape  : {shap_values_gbm.shape}  (samples × features)")
print(f"Base value (E[f(x)]): {base_value_gbm:.4f}")
print(f"\nSHAP values for first test instance:")
pd.Series(shap_values_gbm[0], index=X_test.columns).sort_values().head(10)

### 3.1 Summary plot (dot) — global overview

The dot summary plot shows the **distribution of SHAP values for each feature** across all test instances. Each dot represents one patient.

- **x-axis**: SHAP value — how much the feature pushes the prediction toward benign (+) or malignant (−)
- **colour**: actual feature value (red = high, blue = low)
- **y-axis**: features ranked by mean absolute SHAP value (most important at top)

**Reading the plot:** A cluster of red dots on the right for `worst radius` means high values of `worst radius` strongly push predictions toward malignant — clinically consistent with larger tumours being more likely malignant.

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values_gbm,
    X_test,
    plot_type="dot",
    show=False
)
plt.title("SHAP Summary Plot (GBM) — Impact of each feature on model output",
          fontsize=11, pad=12)
plt.tight_layout()
plt.show()

### 3.2 Summary plot (bar) — mean absolute SHAP importance

The bar chart collapses the per-instance SHAP values into a single global importance score per feature — the **mean absolute SHAP value**. This is a global explanation: it summarises model behaviour across all patients rather than explaining a single prediction.

This is the closest SHAP equivalent to traditional feature importance, but unlike impurity-based importance it is model-agnostic and not biased toward high-cardinality features.

In [ ]:
plt.figure(figsize=(9, 7))
shap.summary_plot(
    shap_values_gbm,
    X_test,
    plot_type="bar",
    show=False
)
plt.title("SHAP Global Feature Importance (GBM) — Mean |SHAP value|",
          fontsize=11, pad=12)
plt.tight_layout()
plt.show()

# Print top 10 features numerically
mean_shap = pd.Series(
    np.abs(shap_values_gbm).mean(axis=0),
    index=X_test.columns
).sort_values(ascending=False)

print("\nTop 10 features by mean |SHAP value|:")
print(mean_shap.head(10).round(4).to_string())

### 3.3 Beeswarm plot

The beeswarm plot is an enhanced version of the dot summary that avoids overplotting by distributing dots horizontally when they would otherwise overlap. It provides a clearer view of the **full distribution** of SHAP values per feature, including bimodal distributions (features that push strongly in both directions for different patients).

In [ ]:
plt.figure(figsize=(10, 8))
shap.plots.beeswarm(
    shap_exp_gbm,
    max_display=15,
    show=False
)
plt.title("SHAP Beeswarm Plot (GBM) — Distribution of feature impacts",
          fontsize=11, pad=12)
plt.tight_layout()
plt.show()

### 3.4 Heatmap

The heatmap displays SHAP values for **all instances simultaneously**, with instances along the x-axis and features along the y-axis. Colour encodes the SHAP value (red = positive, blue = negative). This visualisation is particularly useful for identifying patient subgroups with similar explanation patterns — a cluster of patients with consistently high `worst radius` SHAP values may represent a distinct high-risk subgroup.

In [ ]:
plt.figure(figsize=(13, 7))
shap.plots.heatmap(
    shap_exp_gbm,
    max_display=15,
    show=False
)
plt.title("SHAP Heatmap (GBM) — SHAP values across all test instances",
          fontsize=11, pad=12)
plt.tight_layout()
plt.show()

### 3.5 Waterfall plot — local explanation for a single patient

The waterfall plot explains **a single prediction**. It shows how each feature pushes the model output away from the base value (average prediction) to arrive at the final prediction for this specific patient.

- Bars pointing **right (red)**: feature increases the prediction (pushes toward benign)
- Bars pointing **left (blue)**: feature decreases the prediction (pushes toward malignant)
- The bottom value `E[f(X)]` is the base value; the top value `f(x)` is the final model output

This is a **local explanation** — it is specific to this patient and should not be generalised.

In [ ]:
# Explain the first test instance
patient_idx = 0
true_label  = 'benign' if y_test.iloc[patient_idx] == 1 else 'malignant'
pred_prob   = gbm.predict_proba(X_test.iloc[[patient_idx]])[0, 1]

print(f"Patient {patient_idx} — True label: {true_label} | "
      f"Predicted probability (benign): {pred_prob:.4f}")

plt.figure(figsize=(10, 7))
shap.plots.waterfall(
    shap_exp_gbm[patient_idx],
    max_display=15,
    show=False
)
plt.title(f"SHAP Waterfall Plot (GBM) — Patient {patient_idx} | "
          f"True: {true_label} | P(benign) = {pred_prob:.3f}",
          fontsize=10, pad=12)
plt.tight_layout()
plt.show()

### 3.6 Force plot — compact local explanation

The force plot conveys the same information as the waterfall plot in a horizontal format. Features pushing the prediction **higher** (toward benign) are shown in **red**; features pushing it **lower** (toward malignant) in **blue**. The width of each segment is proportional to the magnitude of the SHAP value.

Force plots are commonly used in clinical XAI papers as a compact, presentation-ready explanation for a single patient case.

In [ ]:
# Static matplotlib version (works in all environments)
shap.force_plot(
    base_value_gbm,
    shap_values_gbm[patient_idx],
    X_test.iloc[patient_idx],
    matplotlib=True,
    show=True,
    figsize=(14, 3)
)

In [ ]:
# Interactive version (requires shap.initjs() and a Jupyter environment)
# Uncomment to use:
# shap.force_plot(
#     base_value_gbm,
#     shap_values_gbm[patient_idx],
#     X_test.iloc[patient_idx]
# )

### 3.7 Dependence plot — feature interaction

The dependence plot shows how the **SHAP value of one feature varies with its actual value**, coloured by a second feature to reveal interactions. This makes it possible to:

- Identify **non-linear relationships** between a feature and model output
- Detect **interaction effects** — does the impact of `worst radius` change depending on `worst concave points`?

SHAP automatically selects the most strongly interacting second feature for colouring if none is specified.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top feature: worst radius
shap.dependence_plot(
    "worst radius",
    shap_values_gbm,
    X_test,
    ax=axes[0],
    show=False
)
axes[0].set_title("Dependence: worst radius", fontsize=10)

# Second feature: worst concave points
shap.dependence_plot(
    "worst concave points",
    shap_values_gbm,
    X_test,
    ax=axes[1],
    show=False
)
axes[1].set_title("Dependence: worst concave points", fontsize=10)

plt.suptitle("SHAP Dependence Plots (GBM) — Feature value vs. SHAP contribution",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

### 3.8 Decision plot — comparing multiple patients

The decision plot traces the cumulative path of SHAP values from the base value to the final prediction for **multiple patients simultaneously**. Each line is one patient; the x-axis shows the running total of SHAP contributions added feature by feature.

This is particularly useful in the medical context for:
- Comparing high-risk vs low-risk patient trajectories
- Identifying where patients with similar outcomes diverge
- Explaining model behaviour to a clinical audience unfamiliar with SHAP

In [ ]:
# Compare first 20 test patients, coloured by true label
n_display  = 20
true_labels = ['benign' if v == 1 else 'malignant'
               for v in y_test.iloc[:n_display].values]

plt.figure(figsize=(11, 7))
shap.decision_plot(
    base_value_gbm,
    shap_values_gbm[:n_display],
    X_test.iloc[:n_display],
    feature_display_range=slice(-1, -16, -1),  # top 15 features
    legend_labels=true_labels,
    show=False
)
plt.title(f"SHAP Decision Plot (GBM) — {n_display} test patients",
          fontsize=11, pad=12)
plt.tight_layout()
plt.show()

<a id='4'></a>
## 4. SHAP with TreeExplainer — Random Forest

TreeExplainer also works with Random Forests. Unlike GBMs (which output a single score for binary classification), the Random Forest `shap_values()` returns a **list of two arrays** — one per class. We extract the SHAP values for class 1 (benign) for visualisation.

Note that the interpretation is the same: positive SHAP values push toward the selected class.

In [ ]:
# ── TreeExplainer for Random Forest ──────────────────────────────────────────
explainer_rf    = shap.TreeExplainer(rf)
shap_values_rf  = explainer_rf.shap_values(X_test)   # shape: (n, features, 2)
shap_rf_class1  = shap_values_rf[:, :, 1]            # SHAP for benign (class 1)
base_value_rf   = explainer_rf.expected_value[1]

print(f"SHAP array shape : {shap_values_rf.shape}  (samples × features × classes)")
print(f"Class-1 SHAP shape: {shap_rf_class1.shape}")
print(f"Base value (benign): {base_value_rf:.4f}")

In [ ]:
# Summary plot for RF — compare with GBM
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plt.sca(axes[0])
shap.summary_plot(shap_values_gbm, X_test, plot_type="bar",
                  show=False, plot_size=None)
axes[0].set_title("GBM — Global feature importance", fontsize=10)

plt.sca(axes[1])
shap.summary_plot(shap_rf_class1, X_test, plot_type="bar",
                  show=False, plot_size=None)
axes[1].set_title("Random Forest — Global feature importance", fontsize=10)

plt.suptitle("SHAP Feature Importance: GBM vs. Random Forest",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Waterfall for a single patient — RF
shap_exp_rf = explainer_rf(X_test)

plt.figure(figsize=(10, 7))
shap.plots.waterfall(
    shap_exp_rf[patient_idx, :, 1],   # class-1 explanation for patient_idx
    max_display=15,
    show=False
)
plt.title(f"SHAP Waterfall (RF) — Patient {patient_idx} | True: {true_label} | "
          f"P(benign) = {rf.predict_proba(X_test.iloc[[patient_idx]])[0,1]:.3f}",
          fontsize=10, pad=12)
plt.tight_layout()
plt.show()

<a id='5'></a>
## 5. SHAP with LinearExplainer — Logistic Regression

`LinearExplainer` computes exact SHAP values for linear models (logistic regression, linear SVM, elastic net, LASSO) by exploiting the additive structure of linear predictions. It is computationally efficient and produces exact — not approximate — attributions.

For logistic regression, SHAP values are computed in **log-odds space** and reflect the marginal contribution of each feature to the log-odds of the positive class, accounting for feature correlations.

> **Important:** Logistic regression requires standardised features for SHAP to produce meaningful coefficient-level attributions. The scaler fitted on the training set is applied here.

In [ ]:
# ── LinearExplainer for Logistic Regression ───────────────────────────────────
explainer_lr   = shap.LinearExplainer(lr, X_train_sc_df)
shap_values_lr = explainer_lr.shap_values(X_test_sc_df)  # (n, features)
base_value_lr  = explainer_lr.expected_value

print(f"SHAP values shape  : {shap_values_lr.shape}")
print(f"Base value (log-odds): {base_value_lr:.4f}")

In [ ]:
# Summary plot — Logistic Regression
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values_lr,
    X_test_sc_df,
    plot_type="dot",
    show=False
)
plt.title("SHAP Summary Plot (Logistic Regression) — Log-odds attributions",
          fontsize=11, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# Force plot — Logistic Regression, single patient
pred_prob_lr = lr.predict_proba(X_test_sc_df.iloc[[patient_idx]])[0, 1]
print(f"Patient {patient_idx} — LR P(benign) = {pred_prob_lr:.4f}")

shap.force_plot(
    base_value_lr,
    shap_values_lr[patient_idx],
    X_test_sc_df.iloc[patient_idx],
    matplotlib=True,
    show=True,
    figsize=(14, 3)
)

In [ ]:
# Compare SHAP rankings: GBM vs LR
rank_gbm = pd.Series(
    np.abs(shap_values_gbm).mean(axis=0),
    index=X_test.columns
).rank(ascending=False).astype(int)

rank_lr = pd.Series(
    np.abs(shap_values_lr).mean(axis=0),
    index=X_test.columns
).rank(ascending=False).astype(int)

comparison = pd.DataFrame({
    'GBM rank':  rank_gbm,
    'LR rank':   rank_lr,
    'Rank diff': (rank_gbm - rank_lr).abs()
}).sort_values('GBM rank')

print("Feature importance rank comparison (GBM vs Logistic Regression):")
print(comparison.head(15).to_string())

<a id='6'></a>
## 6. Summary: SHAP explainer types and when to use them

The table below summarises the SHAP explainers demonstrated in this notebook and provides practical guidance for selecting the appropriate explainer for a given clinical ML model.

In [ ]:
summary = pd.DataFrame([
    {
        "Explainer"        : "TreeExplainer",
        "Compatible models": "Decision Tree, Random Forest, GBM, XGBoost, LightGBM, CatBoost",
        "Speed"            : "Fast (exact)",
        "Explanation type" : "Model-specific, exact",
        "Best for"         : "Tabular EHR data; the dominant workflow in medical XAI"
    },
    {
        "Explainer"        : "LinearExplainer",
        "Compatible models": "Logistic Regression, Linear SVM, ElasticNet, LASSO, Ridge",
        "Speed"            : "Fast (exact)",
        "Explanation type" : "Model-specific, exact",
        "Best for"         : "Interpretable baseline models; clinical risk scores"
    },
    {
        "Explainer"        : "DeepExplainer",
        "Compatible models": "TensorFlow / Keras / PyTorch neural networks",
        "Speed"            : "Medium (approximate)",
        "Explanation type" : "Model-specific, approximate",
        "Best for"         : "Deep learning on tabular or sequence data; EHR time-series"
    },
    {
        "Explainer"        : "GradientExplainer",
        "Compatible models": "TensorFlow / Keras / PyTorch neural networks",
        "Speed"            : "Medium (approximate)",
        "Explanation type" : "Model-specific, approximate",
        "Best for"         : "CNN-based medical image models as alternative to Grad-CAM"
    },
    {
        "Explainer"        : "KernelExplainer",
        "Compatible models": "Any black-box model (SVM, MLP, ensembles, etc.)",
        "Speed"            : "Slow (approximate, sampling-based)",
        "Explanation type" : "Model-agnostic, approximate",
        "Best for"         : "Models without a specialised explainer; small test sets"
    },
])

print(summary.to_string(index=False))

In [ ]:
# ── SHAP plot types at a glance ───────────────────────────────────────────────
plot_guide = pd.DataFrame([
    {"Plot type"     : "Summary (dot)",
     "Scope"         : "Global",
     "SHAP function" : "shap.summary_plot(..., plot_type='dot')",
     "Use case"      : "Overview of feature impact directions across all patients"},
    {"Plot type"     : "Summary (bar)",
     "Scope"         : "Global",
     "SHAP function" : "shap.summary_plot(..., plot_type='bar')",
     "Use case"      : "Rank features by overall importance; replaces impurity importance"},
    {"Plot type"     : "Beeswarm",
     "Scope"         : "Global",
     "SHAP function" : "shap.plots.beeswarm(explanation)",
     "Use case"      : "Detailed per-patient distribution; avoids overplotting"},
    {"Plot type"     : "Heatmap",
     "Scope"         : "Global",
     "SHAP function" : "shap.plots.heatmap(explanation)",
     "Use case"      : "Identify patient subgroups with similar explanation patterns"},
    {"Plot type"     : "Waterfall",
     "Scope"         : "Local",
     "SHAP function" : "shap.plots.waterfall(explanation[i])",
     "Use case"      : "Explain a single patient prediction step-by-step"},
    {"Plot type"     : "Force plot",
     "Scope"         : "Local",
     "SHAP function" : "shap.force_plot(base, values, features)",
     "Use case"      : "Compact single-patient explanation; used in CDSS interfaces"},
    {"Plot type"     : "Dependence",
     "Scope"         : "Global",
     "SHAP function" : "shap.dependence_plot(feature, values, X)",
     "Use case"      : "Visualise non-linear effects and feature interactions"},
    {"Plot type"     : "Decision",
     "Scope"         : "Local + comparative",
     "SHAP function" : "shap.decision_plot(base, values, features)",
     "Use case"      : "Compare explanation paths across multiple patients"},
])

print(plot_guide.to_string(index=False))

---

## Key takeaways

1. **SHAP values are additive**: `f(x) = E[f(X)] + Σ φᵢ` — the prediction equals the base value plus the sum of all feature contributions. This property makes SHAP mathematically rigorous and directly interpretable.

2. **Choose the right explainer for your model**: `TreeExplainer` for tree ensembles (fast, exact), `LinearExplainer` for linear models (fast, exact), `DeepExplainer`/`GradientExplainer` for neural networks (approximate), `KernelExplainer` for any black-box model (slow, approximate).

3. **Global ≠ local**: A feature that ranks first in global importance may have near-zero SHAP value for a specific patient. Always use local plots (waterfall, force) when explaining individual clinical decisions.

4. **SHAP values are not causal**: A high SHAP value indicates that a feature's value contributed to this prediction in this model — it does not mean the feature *causes* the outcome. This distinction is especially important in clinical contexts where the explanation may influence treatment decisions.

5. **Stability matters**: SHAP values can vary across model runs, dataset splits, and small input perturbations. For clinical deployment, always report confidence intervals or bootstrap ranges alongside point SHAP estimates.

---

## Further reading

- Lundberg, S.M. & Lee, S.-I. (2017). *A unified approach to interpreting model predictions*. NeurIPS. — The original SHAP paper.
- Lundberg, S.M. et al. (2020). *From local explanations to global understanding with explainable AI for trees*. Nature Machine Intelligence. — TreeSHAP and clinical applications.
- SHAP documentation: https://shap.readthedocs.io
- SHAP GitHub: https://github.com/shap/shap